# 02 — Inference with HIVRoBERTa (from the HF model repo)
Classify your own messages. Pulls the released checkpoint and its learned thresholds from the model repo; `config.py` supplies credentials (env vars take precedence).

**About neutral:** the model outputs seven probabilities and has **no neutral class** — in the paper's system, neutral is decided by the LLM refinement stage (final cells). The backbone-only cells below apply the paper's *standalone scoring convention* — the highest-probability emotion among those crossing their thresholds, neutral when none crosses — solely so their outputs are comparable to the paper's backbone-only tables. Do **not** replace the shipped thresholds with 0.5.

In [ ]:
%%capture
!pip install -q "transformers==4.57.1" huggingface_hub torch numpy pandas openai

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download
import config
tokenizer = AutoTokenizer.from_pretrained(config.HF_MODEL_REPO, token=config.get_hf_token())
model = AutoModelForSequenceClassification.from_pretrained(config.HF_MODEL_REPO, token=config.get_hf_token()).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device)
thr = np.load(hf_hub_download(config.HF_MODEL_REPO, "best_thr_f1.npy", token=config.get_hf_token()))
TARGET_EMOTIONS = [model.config.id2label[i] for i in range(model.config.num_labels)]
print("labels:", TARGET_EMOTIONS, "\nthresholds:", dict(zip(TARGET_EMOTIONS, thr.round(2))))

In [ ]:
def predict_emotions(text, tokenizer, model, thresholds, device):
    enc = tokenizer(
        text, return_tensors="pt", truncation=True,
        padding="max_length", max_length=MAX_LENGTH,
    ).to(device)
    logits = model(**enc).logits.cpu().numpy()[0]
    probs = 1 / (1 + np.exp(-logits))
    binary = (probs >= thresholds).astype(int)

    active = np.where(binary == 1)[0]
    final = "neutral" if len(active) == 0 else TARGET_EMOTIONS[active[np.argmax(probs[active])]]

    return {
        "probs": {e: float(p) for e, p in zip(TARGET_EMOTIONS, probs)},
        "preds": {e: int(b) for e, b in zip(TARGET_EMOTIONS, binary)},
        "final_label": final,
    }

# Backbone-only, using the paper's standalone convention (no crossing -> neutral).
# The model itself never predicts neutral; the deployed pipeline's neutral comes from the LLM stage below.
def classify(texts):
    return [predict_emotions(t, tokenizer, model, thr, device) for t in texts]
demo = ["I get scared every time my refill is late.",
        "They moved my appointment again without asking and I am done being polite.",
        "Just confirming the time for tomorrow."]
for t, out in zip(demo, classify(demo)):
    print(f"{out['final_label']:13s} | {t}")

## Your own data — a list of strings or a CSV with a `text` column

In [ ]:
import pandas as pd
# texts = ["...", "..."]                                     # option A: a list
# texts = pd.read_csv("my_messages.csv")["text"].tolist()    # option B: a CSV
texts = demo
rows=[]
for t, out in zip(texts, classify(texts)):
    rows.append({"text": t, "label": out["final_label"], **{f"p_{e}": round(p,3) for e,p in out["probs"].items()}})
pred = pd.DataFrame(rows); pred.to_csv("predictions.csv", index=False); pred

## The full pipeline — backbone + GPT-5-mini, one function
`classify_pipeline(texts)` runs both stages and writes `predictions_pipeline.csv` with the final label, the backbone's own label, per-emotion probabilities, and the model's short explanation.

In [ ]:
# The paper's FULL pipeline in one call: backbone probabilities -> GPT-5-mini refinement -> final label.
# Needs the OpenAI key in config.py or the OPENAI_API_KEY env var. A failed or unparseable call yields
# label=None for that row (no substituted labels — neutral only ever comes from the LLM's own decision).
import json
import pandas as pd
from openai import OpenAI
import config

REFINEMENT_MODEL = "gpt-5-mini"

def build_llm_prompt(user_text: str, backbone_output: dict) -> str:
    probs_str = "\n".join(
        f"- {e}: {backbone_output['probs'][e]:.3f}" for e in TARGET_EMOTIONS
    )

    return f"""
You are an expert clinical annotator helping label emotional content in short, user-written messages.
You work with EXACTLY these 8 possible labels:

- anger
- disgust
- embarrassment
- fear
- grief
- nervousness
- sadness
- neutral   (no clear or dominant negative emotion)

You are given:
1) A message written by a person.
2) The output of an emotion classifier (M1), which provides probabilities for the 7 emotions above
   (anger, disgust, embarrassment, fear, grief, nervousness, sadness).
   If none of these emotions clearly fits the message, the most appropriate label is "neutral".

Your task:
- Carefully read the message.
- Use the classifier's probabilities as a hint, but you may OVERRIDE them if they don't match the text.
- Choose EXACTLY ONE final label from the 8 above.
- Briefly justify your choice in 1-3 sentences.
- Be especially careful to distinguish:
  * sadness vs grief (grief is usually linked to loss, death, or bereavement),
  * fear vs nervousness (fear = strong acute danger; nervousness = milder anxiety, worry, anticipation),
  * anger vs disgust (anger = attack/hostility; disgust = moral/physical revulsion),
  * embarrassment (social awkwardness, shame, wanting to hide) vs the others.
- "neutral" means the message is mostly informational or very mild in emotion.

Always answer in VALID JSON, with this exact structure (no extra keys, no extra text):

{{
  "final_label": "<one of: anger, disgust, embarrassment, fear, grief, nervousness, sadness, neutral>",
  "explanation": "<your short explanation>"
}}

--------------------------
FEW-SHOT EXAMPLES
--------------------------

Example 1
Message:
\"\"\"I've been crying all day since I got my HIV diagnosis. I feel completely empty and I don't see how I'll ever be happy again.\"\"\"

M1 emotion probabilities:
- anger: 0.041
- disgust: 0.018
- embarrassment: 0.072
- fear: 0.452
- grief: 0.731
- nervousness: 0.311
- sadness: 0.845

Assistant JSON answer:
{{
  "final_label": "sadness",
  "explanation": "The message expresses intense sadness and despair about the diagnosis. Although there are elements of fear and grief, the dominant tone is pervasive sadness and hopelessness."
}}

--------------------------

Example 2
Message:
\"\"\"People act like I'm dirty just because I'm HIV positive. I'm so sick of their judgment, it makes me want to scream.\"\"\"

M1 emotion probabilities:
- anger: 0.812
- disgust: 0.644
- embarrassment: 0.221
- fear: 0.093
- grief: 0.051
- nervousness: 0.102
- sadness: 0.303

Assistant JSON answer:
{{
  "final_label": "anger",
  "explanation": "The speaker is reacting to unfair treatment and judgment with a strong desire to 'scream', which indicates anger more than disgust or sadness."
}}

--------------------------

Example 3
Message:
\"\"\"My HIV test results come back tomorrow and my stomach has been in knots all day. I can't stop thinking about it.\"\"\"

M1 emotion probabilities:
- anger: 0.035
- disgust: 0.010
- embarrassment: 0.047
- fear: 0.521
- grief: 0.022
- nervousness: 0.781
- sadness: 0.166

Assistant JSON answer:
{{
  "final_label": "nervousness",
  "explanation": "The message focuses on anticipation, worry, and physical signs of anxiety about future results, which fits nervousness better than full-blown fear."
}}

--------------------------

Example 4
Message:
\"\"\"I have an appointment with my doctor next week to adjust my HIV medication. They said we'll review my latest lab results.\"\"\"

M1 emotion probabilities:
- anger: 0.012
- disgust: 0.008
- embarrassment: 0.014
- fear: 0.042
- grief: 0.003
- nervousness: 0.089
- sadness: 0.054

Assistant JSON answer:
{{
  "final_label": "neutral",
  "explanation": "The message is mainly factual and logistical about an upcoming appointment, without a clear negative emotional tone."
}}

--------------------------
NOW ANALYZE THIS NEW MESSAGE
--------------------------

Message:
\"\"\"{user_text}\"\"\"

M1 emotion probabilities:
{probs_str}

Remember:
- You must choose exactly ONE label from: anger, disgust, embarrassment, fear, grief, nervousness, sadness, neutral.
- Use M1 probabilities as a hint, but override them if the text suggests a different emotion.
- Respond ONLY with valid JSON in the specified format.

Assistant JSON answer:
""".strip()

def call_llm(client, prompt: str) -> str:
    """gpt-5-mini rejects the temperature parameter, so it is not sent."""
    try:
        resp = client.chat.completions.create(
            model=REFINEMENT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an assistant that ALWAYS responds with a single JSON "
                        "object. This message explicitly contains the word json to "
                        "satisfy the API requirement."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            response_format={"type": "json_object"},
        )
        return resp.choices[0].message.content
    except Exception as exc:
        return json.dumps({"final_label": None, "explanation": f"call failed: {exc}"})


# --------------------------------------------------------------------------- #

_client = None
def classify_pipeline(texts, include_explanations=True):
    """Run the complete two-stage pipeline on a list of texts. Returns a DataFrame."""
    global _client
    if _client is None:
        _client = OpenAI(api_key=config.get_openai_key())
    valid = set(TARGET_EMOTIONS) | {"neutral"}
    rows = []
    for t in texts:
        bo = predict_emotions(t, tokenizer, model, thr, device)
        raw = call_llm(_client, build_llm_prompt(t, bo))
        try:
            parsed = json.loads(raw)
            label, expl = parsed.get("final_label"), parsed.get("explanation", "")
        except Exception:
            label, expl = None, f"unparseable response: {raw[:120]}"
        if label not in valid:
            label = None   # no substitution — the pipeline's label comes from the LLM or not at all
            expl = (expl + " [call failed or invalid — rerun this row]").strip()
        rows.append({"text": t, "label": label, "backbone_label": bo["final_label"],
                     "explanation": expl if include_explanations else "",
                     **{f"p_{e}": round(bo["probs"][e], 3) for e in TARGET_EMOTIONS}})
    df = pd.DataFrame(rows)
    n_fail = int(df["label"].isna().sum())
    if n_fail: print(f"WARNING: {n_fail} row(s) have no label (LLM call failed) — rerun those rows.")
    return df

pipe = classify_pipeline(texts)      # `texts` from the cell above — a list or your CSV column
pipe.to_csv("predictions_pipeline.csv", index=False)
pipe[["label", "backbone_label", "text"]]